# Raw Reviews Data (Table View)
Source: `data/raw_data/reviews.json`


In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Resolve repo root whether notebook lives at repo root or notebooks/
NOTEBOOK_DIR = os.path.abspath('')
if os.path.exists(os.path.join(NOTEBOOK_DIR, 'config.py')):
    PROJECT_DIR = NOTEBOOK_DIR
else:
    PROJECT_DIR = os.path.dirname(NOTEBOOK_DIR)

APP_NAME = 'RawReviewsData'
DATA_PATH = os.path.join(PROJECT_DIR, 'data/raw_data/reviews.json')

spark = (
    SparkSession.builder
    .appName(APP_NAME)
    .master('local[*]')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

df_raw = spark.read.option('multiLine', 'true').json(DATA_PATH)

df_exploded = df_raw.select(
    F.col('id').alias('property_id'),
    F.explode('reviews').alias('review')
)

df_view = df_exploded.select(
    F.col('property_id'),
    F.col('review.id').alias('review_id'),
    F.col('review.date').alias('review_date'),
    F.col('review.score').alias('review_score'),
    F.col('review.language').alias('review_language'),
    F.col('review.reviewer.name').alias('reviewer_name'),
    F.col('review.reviewer.country').alias('reviewer_country'),
    F.col('review.summary').alias('review_summary')
)

df_view.show(20, truncate=False)

spark.createDataFrame(df_view.dtypes, ['column', 'dtype']).show(truncate=False)

df_view.count()
